In [1]:
!pip install gymnasium ale-py autorom

In [2]:
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	C:\Users\zspicher\AppData\Local\anaconda3\Lib\site-packages\AutoROM\roms

Existing ROMs will be overwritten.


C:\Users\zspicher\AppData\Local\anaconda3\Lib\site-packages\AutoROM\AutoROM.py:264: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
!pip install pygame

In [4]:
!pip install opencv-python

In [7]:
%pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.3/114.6 MB 9.8 MB/s eta 0:00:12
   - -------------------------------------- 4.5/114.6 MB 13.2 MB/s eta 0:00:09
   --- ------------------------------------ 8.7/114.6 MB 15.9 MB/s eta 0:00:07
   ---- ----------------------------------- 12.6/114.6 MB 16.8 MB/s eta 0:00:07
   ----- ---------------------------------- 16.8/114.6 MB 17.0 MB/s eta 0:00:06
   ------- -------------------------------- 21.8/114.6 MB 18.0 MB/s eta 0:00:06
   --------- ------------------------------ 26.2/114.6 MB 18.3 MB/s eta 0:00:05
   ---------- ----------------------------- 29.4/114.6 MB 18.0 MB/s eta 0:00:05
   ----------- ---------------------------- 32.5/114.6 MB 17.4 MB/s eta 0:00:05
   ------------ --------------------------- 34.6/114.6 MB 16.8 MB/s eta 0:00:05
   ------------- -------------------------- 38.0/114.6 MB 16.6 MB/s eta 0:00:05
   -------------- ------------------------- 41.7/114.

In [10]:
import gymnasium as gym
import ale_py
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import torch
import torch.nn as nn

In [11]:
gym.register_envs(ale_py)

# Visual env (for drawing)
env_vis = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

# RAM env (for logic)
env_ram = gym.make("ALE/Freeway-v5", obs_type="ram")

# Reset both environments to get initial observations
obs_vis, _ = env_vis.reset()
obs_ram, _ = env_ram.reset()

In [12]:

# ---------------------------
# MODEL
# ---------------------------
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_size)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(128, env.action_space.n)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
# ---------------------------
# MODEL INITIALIZATION
# ---------------------------
# Input size = RAM size (128 bytes)
input_size = 128

# Number of possible actions in the environment
output_size = env_vis.action_space.n

# Create the neural network model
model = MLP(input_size, output_size)


In [15]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

log_probs = []
rewards = []

prev_y = None

In [ ]:

# ---------------------------
# TRAINING LOOP
# ---------------------------
episodes = 50  # small for fast testing

for episode in range(episodes):
    obs, _ = env.reset()

    log_probs = []
    rewards = []
    prev_y = None

    done = False

    while not done:
        ram_input = torch.tensor(obs, dtype=torch.float32).unsqueeze(0) / 255.0

        logits = model(ram_input)
        probs = torch.softmax(logits, dim=1)

        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        log_prob = dist.log_prob(action)

        obs, _, term, trunc, _ = env.step(action.item())
        done = term or trunc

        # --- reward ---
        chicken_y = int(obs[14])
        if prev_y is None:
            reward = 0
        else:
            reward = chicken_y - prev_y

        prev_y = chicken_y

        log_probs.append(log_prob)
        rewards.append(reward)

    # --- TRAIN ---
    loss = 0
    for log_prob, reward in zip(log_probs, rewards):
        loss += -log_prob * reward

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {episode} | Total reward: {sum(rewards)}")

# ---------------------------
# SAVE MODEL
# ---------------------------
torch.save(model.state_dict(), "freeway_mlp.pth")

env.close()

In [ ]:
torch.save(model.state_dict(), "freeway_mlp.pth")
env.close()

In [14]:

# ---------------------------
# CREATE VISUAL ENVIRONMENTS
# ---------------------------
env_vis = gym.make("ALE/Freeway-v5", render_mode="rgb_array")
env_ram = gym.make("ALE/Freeway-v5", obs_type="ram")

obs_vis, _ = env_vis.reset()
obs_ram, _ = env_ram.reset()

# ---------------------------
# LOAD TRAINED MODEL
# ---------------------------
model.load_state_dict(torch.load("freeway_mlp.pth"))
model.eval()

# ---------------------------
# PLAY LOOP
# ---------------------------
while True:

    # prepare input
    ram_input = np.array(obs_ram, dtype=np.float32) / 255.0
    ram_input = torch.tensor(ram_input).unsqueeze(0)

    # use learned behavior (no randomness)
    with torch.no_grad():
        logits = model(ram_input)
        action = torch.argmax(logits, dim=1).item()

    # step environment
    obs_vis, _, term1, trunc1, _ = env_vis.step(action)
    obs_ram, _, term2, trunc2, _ = env_ram.step(action)

    frame = obs_vis.copy()

    # display
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    cv2.imshow("Freeway AI", frame_bgr)
    time.sleep(0.03)

    # quit key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    # reset instead of stopping
    if term1 or trunc1:
        obs_vis, _ = env_vis.reset()
        obs_ram, _ = env_ram.reset()

env_vis.close()
env_ram.close()

KeyboardInterrupt: 